# <font color="#418FDE" size="6.5" uppercase>**Merkmale erklären**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Konstruieren und wählen Merkmale innerhalb von scikit-learn-Pipelines aus. 
- Vergleichen Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen. 
- Interpretieren Modelle mit Koeffizienten, Importances und Abhängigkeitsplots. 


## **1. Merkmale konstruieren**

### **1.1. Polynomiale Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_01.jpg?v=1787648046" width="250">



>* Polynomiale Merkmale modellieren gekrümmte Zusammenhänge
>* Pipelines erzeugen sie sauber und reproduzierbar

>* Interaktionen zwischen Merkmalen werden modellierbar
>* Komplexere Muster, aber schwierigere Interpretation

>* Mehr Polynomgrade erhöhen Überanpassungsrisiko.
>* Mit Skalierung, Regularisierung und Validierung prüfen.



In [ ]:
#@title Python-Code - Polynomiale Merkmale

# Dieses Beispiel zeigt polynomiale Merkmale in Pipelines.
# Ein lineares Modell lernt dadurch gekrümmte Muster.
# Der Plot vergleicht einfache und erweiterte Merkmale.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine, gekrümmte Regressionsdaten.
rng = np.random.default_rng(42)
x = np.linspace(-3.0, 3.0, 80)
noise = rng.normal(0.0, 1.2, size=x.shape)
y = 2.0 + 0.8 * x - 1.4 * x**2 + noise

# Scikit-learn erwartet eine zweidimensionale Merkmalsmatrix.
X = x.reshape(-1, 1)
if X.shape != (80, 1):
    raise ValueError("Die Merkmalsmatrix hat nicht die erwartete Form.")

# Dieses Modell nutzt nur das ursprüngliche Merkmal.
linear_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
linear_model.fit(X, y)
linear_predictions = linear_model.predict(X)

# Diese Pipeline konstruiert zusätzlich x² und x³.
poly_model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.0),
)
poly_model.fit(X, y)
poly_predictions = poly_model.predict(X)

# Die Namen zeigen die konstruierten Merkmale.
feature_step = poly_model.named_steps["polynomialfeatures"]
feature_names = feature_step.get_feature_names_out(["x"])
linear_rmse = mean_squared_error(y, linear_predictions) ** 0.5
poly_rmse = mean_squared_error(y, poly_predictions) ** 0.5

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Konstruierte Merkmale: {', '.join(feature_names)}")
print(f"RMSE ohne polynomiale Merkmale: {linear_rmse:.2f}")
print(f"RMSE mit polynomialen Merkmalen: {poly_rmse:.2f}")

# Der Plot macht die bessere Anpassung sichtbar.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[:, 0], y, s=25, alpha=0.7, label="Datenpunkte")
ax.plot(X[:, 0], linear_predictions, label="nur x", linewidth=2)
ax.plot(X[:, 0], poly_predictions, label="x, x², x³", linewidth=2)

ax.set_title("Polynomiale Merkmale in einer scikit-learn-Pipeline")
ax.set_xlabel("Eingangsmerkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



### **1.2. Binning und Splines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_02.jpg?v=1787648050" width="250">



>* Binning wandelt Zahlen in sinnvolle Kategorien um
>* Lineare Modelle erfassen dadurch Schwellen besser

>* Bins bewusst nach Daten und Fachwissen wählen
>* Pipeline verhindert Datenleckage durch Trainingsgrenzen

>* Splines modellieren glatte nichtlineare Zusammenhänge
>* Pipelines bewerten Spline-Merkmale reproduzierbar



In [ ]:
#@title Python-Code - Binning und Splines

# Dieses Beispiel vergleicht lineare, gebinnte und Spline-Merkmale.
# Pipelines lernen Umwandlungen nur aus Trainingsdaten.
# Die Kurven zeigen unterschiedliche Modellflexibilität.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import SplineTransformer
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine, nichtlineare Regressionsdaten.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 160).reshape(-1, 1)
y = np.sin(x[:, 0]) + 0.25 * x[:, 0] + rng.normal(0, 0.18, 160)

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if x.shape != (160, 1) or y.shape != (160,):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Der Split trennt Training und Test vor jeder Umwandlung.
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

# Drei Pipelines nutzen dasselbe Modell, aber andere Merkmale.
linear_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
binned_model = make_pipeline(
    KBinsDiscretizer(n_bins=8, encode="onehot-dense", strategy="quantile"),
    Ridge(alpha=1.0),
)

# Splines erzeugen glatte Basisfunktionen für dasselbe Merkmal.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3, include_bias=False),
    Ridge(alpha=1.0),
)

# Alle Pipelines werden nur auf den Trainingsdaten angepasst.
linear_model.fit(x_train, y_train)
binned_model.fit(x_train, y_train)
spline_model.fit(x_train, y_train)

# Die Testfehler vergleichen die konstruierten Merkmale fair.
linear_rmse = mean_squared_error(y_test, linear_model.predict(x_test)) ** 0.5
binned_rmse = mean_squared_error(y_test, binned_model.predict(x_test)) ** 0.5
spline_rmse = mean_squared_error(y_test, spline_model.predict(x_test)) ** 0.5

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"RMSE linear: {linear_rmse:.3f}")
print(f"RMSE mit Binning: {binned_rmse:.3f}")
print(f"RMSE mit Splines: {spline_rmse:.3f}")

# Ein feines Raster zeigt die gelernten Kurven.
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)
linear_pred = linear_model.predict(x_grid)
binned_pred = binned_model.predict(x_grid)
spline_pred = spline_model.predict(x_grid)

# Eine einzige Grafik macht harte und glatte Übergänge sichtbar.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train[:, 0], y_train, s=18, alpha=0.45, label="Trainingsdaten")
ax.plot(x_grid[:, 0], linear_pred, label="Linear")
ax.plot(x_grid[:, 0], binned_pred, label="Binning")
ax.plot(x_grid[:, 0], spline_pred, label="Splines")

ax.set_title("Merkmalskonstruktion in scikit-learn-Pipelines")
ax.set_xlabel("Eingabemerkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



### **1.3. Varianz und KBest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_03.jpg?v=1787648048" width="250">



>* Merkmale mit geringer Varianz entfernen
>* Auswahl nur innerhalb der Pipeline lernen

>* Varianz ignoriert Zielvariable und Vorhersagerelevanz
>* Nützlich als grober Pipeline-Filter

>* KBest wählt zielbezogen die stärksten Merkmale
>* Pipeline-Validierung beachten, Wechselwirkungen bleiben begrenzt



In [ ]:
#@title Python-Code - Varianz und KBest

# Dieses Beispiel zeigt Merkmalsauswahl in einer Pipeline.
# Varianzfilter entfernen zuerst fast konstante Spalten.
# KBest wählt danach zielbezogen die stärksten Merkmale.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine Klassifikationsdaten mit festen Zufallswerten.
X, y = make_classification(
    n_samples=300, n_features=6, n_informative=3, n_redundant=0,
    n_repeated=0, n_classes=2, random_state=42, shuffle=False
)

# Zwei zusätzliche Spalten sind konstant oder fast konstant.
constant_column = np.ones((X.shape[0], 1))
rare_column = (np.arange(X.shape[0]) == 0).astype(float).reshape(-1, 1)
X = np.hstack([X, constant_column, rare_column])

# Ein kurzer Check macht die erwartete Tabellenform sichtbar.
if X.shape != (300, 8):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Die Aufteilung verhindert Datenleckage bei Auswahl und Skalierung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Die Pipeline lernt alle Auswahlregeln nur aus Trainingsdaten.
pipeline = Pipeline([
    ("variance", VarianceThreshold(threshold=0.01)),
    ("kbest", SelectKBest(score_func=f_classif, k=3)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])

# Ein einziges Training reicht für diese fokussierte Demonstration.
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

# Wir rekonstruieren, welche ursprünglichen Spalten übrig bleiben.
feature_names = np.array([
    "Merkmal 0", "Merkmal 1", "Merkmal 2", "Merkmal 3",
    "Merkmal 4", "Merkmal 5", "konstant", "selten"
])

# Erst entfernt der Varianzfilter sehr schwache Spalten.
variance_mask = pipeline.named_steps["variance"].get_support()
after_variance = feature_names[variance_mask]

# Danach wählt KBest die drei zielbezogen besten Spalten.
kbest_mask = pipeline.named_steps["kbest"].get_support()
selected_names = after_variance[kbest_mask]

# Die KBest-Werte zeigen die Stärke der Einzelbeziehung.
scores = pipeline.named_steps["kbest"].scores_
selected_scores = scores[kbest_mask]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Startmerkmale: {X.shape[1]}")
print(f"Nach Varianzfilter: {len(after_variance)}")
print(f"Nach KBest: {len(selected_names)}")
print(f"Testgenauigkeit: {accuracy:.2f}")
print("Ausgewählt: " + ", ".join(selected_names))

# Das Balkendiagramm zeigt die KBest-Bewertungen der Auswahl.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(selected_names, selected_scores, color="steelblue")
ax.set_title("KBest-Bewertungen nach Varianzfilter")
ax.set_xlabel("Ausgewählte Merkmale")
ax.set_ylabel("F-Wert")
plt.show()



## **2. Auswahl und Reduktion**

### **2.1. Rekursive Merkmalsauswahl**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_01.jpg?v=1787648060" width="250">



>* Modell entfernt schrittweise schwache Merkmale
>* Bewertet Merkmale im gemeinsamen Zusammenspiel

>* Kompaktere Modelle werden erklärbarer und robuster
>* Wichtige Merkmale hängen vom Modell ab

>* Originalmerkmale bleiben verständlich und nachvollziehbar
>* Benötigt Rechenzeit und saubere Validierung



In [ ]:
#@title Python-Code - Rekursive Merkmalsauswahl

# Dieses Beispiel zeigt rekursive Merkmalsauswahl praktisch.
# RFE entfernt schrittweise weniger hilfreiche Merkmale.
# Der Plot vergleicht Auswahl und Modellleistung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung schützt den Testdatensatz vor Datenleckage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# RFE nutzt ein lineares Modell zur wiederholten Merkmalsbewertung.
base_model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)
selector = RFE(estimator=base_model, n_features_to_select=8, step=2)

# Die Pipeline skaliert nur mit Trainingsdaten.
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("selector", selector), ("model", base_model)]
)

# Jetzt trainieren wir Auswahl und Modell gemeinsam.
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Wir lesen die ausgewählten ursprünglichen Merkmalsnamen aus.
selected_mask = pipeline.named_steps["selector"].support_
selected_names = data.feature_names[selected_mask]

# Die Rangwerte zeigen, welche Merkmale RFE bevorzugt.
ranking = pipeline.named_steps["selector"].ranking_
selected_ranks = ranking[selected_mask]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit mit 8 Merkmalen: {accuracy:.3f}")
print("Ausgewählte Merkmale:")
for name in selected_names[:5]:
    print(f"- {name}")

# Der Plot zeigt die ausgewählten Merkmale als Rang eins.
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(selected_names))
ax.bar(positions, selected_ranks, color="steelblue")

ax.set_title("Von RFE ausgewählte Merkmale")
ax.set_xlabel("Ausgewähltes Merkmal")
ax.set_ylabel("RFE-Rang")
ax.set_xticks(positions)

ax.set_xticklabels(selected_names, rotation=45, ha="right")
ax.set_ylim(0, 2)
fig.tight_layout()
plt.show()



### **2.2. PCA und SVD**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_02.jpg?v=1787648057" width="250">



>* PCA bündelt korrelierte Zahlenmerkmale in Hauptkomponenten
>* Weniger Dimensionen, aber geringere direkte Interpretierbarkeit

>* SVD eignet sich für riesige, dünne Daten.
>* Kompakte Räume sparen Rechenzeit und zeigen Muster.

>* Datenform bestimmt PCA, SVD oder Auswahl
>* Vergleiche Leistung, Deutbarkeit und Ressourcen



In [ ]:
#@title Python-Code - PCA und SVD

# Dieses Beispiel vergleicht PCA und SVD.
# Beide Verfahren verdichten viele korrelierte Merkmale.
# Die Grafik zeigt erhaltene Varianzanteile.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine korrelierte Messdaten ohne Download.
rng = np.random.default_rng(42)
base = rng.normal(size=(120, 2))
noise = rng.normal(scale=0.15, size=(120, 4))

# Vier Merkmale entstehen aus zwei gemeinsamen Grundmustern.
feature_1 = base[:, 0] + noise[:, 0]
feature_2 = 0.9 * base[:, 0] + noise[:, 1]
feature_3 = base[:, 1] + noise[:, 2]
feature_4 = 0.8 * base[:, 1] + noise[:, 3]

# Die Matrix hat viele Spalten, aber nur zwei Hauptmuster.
X = np.column_stack((feature_1, feature_2, feature_3, feature_4))
if X.shape != (120, 4):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# PCA arbeitet typischerweise auf skalierten dichten Zahlenwerten.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=4, random_state=42)

# SVD wird hier auf derselben Matrix zum Vergleich genutzt.
svd = TruncatedSVD(n_components=4, random_state=42)
pca.fit(X_scaled)
svd.fit(X_scaled)

# Kumulative Anteile zeigen, wie kompakt die Darstellung ist.
pca_cumulative = np.cumsum(pca.explained_variance_ratio_)
svd_cumulative = np.cumsum(svd.explained_variance_ratio_)
components = np.arange(1, 5)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Datenform: {X.shape[0]} Zeilen und {X.shape[1]} Merkmale")
print(f"PCA mit 2 Komponenten erklärt: {pca_cumulative[1]:.2%}")
print(f"SVD mit 2 Komponenten erklärt: {svd_cumulative[1]:.2%}")
print("Merksatz: PCA/SVD erzeugen neue Komponenten, keine Originalspalten.")

# Die Kurven vergleichen die Verdichtung der Information.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(components, pca_cumulative, marker="o", label="PCA")
ax.plot(components, svd_cumulative, marker="s", label="SVD")

ax.set_title("Kumulative erklärte Varianz bei PCA und SVD")
ax.set_xlabel("Anzahl der Komponenten")
ax.set_ylabel("Erklärter Varianzanteil")
ax.set_xticks(components)

ax.set_ylim(0, 1.05)
ax.legend()
plt.show()



### **2.3. Auswahl in Pipelines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_03.jpg?v=1787648062" width="250">



>* Pipelines machen Verarbeitungsschritte reproduzierbar.
>* Merkmalsauswahl nur auf Trainingsdaten lernen.

>* Tabellen: Merkmale wählen, Bedeutung erhalten
>* Hochdimensionale Daten: reduzieren und Pipeline-Vergleiche nutzen

>* Auswahl, Reduktion und Hyperparameter gemeinsam abstimmen
>* Pipelines sichern faire Tests und Wiederverwendung



In [ ]:
#@title Python-Code - Auswahl in Pipelines

# Wir vergleichen Merkmalsauswahl direkt in einer Pipeline.
# Die Auswahl wird nur auf Trainingsdaten gelernt.
# Das Ergebnis zeigt Genauigkeit und gewählte Merkmale.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine tabellarische Klassifikationsdaten.
X, y = make_classification(
    n_samples=300,
    n_features=12,
    n_informative=4,
    n_redundant=2,
    random_state=42,
)

# Verständliche Spaltennamen helfen bei der Interpretation.
feature_names = np.array([f"Merkmal {number}" for number in range(1, 13)])
if X.shape[1] != len(feature_names):
    raise ValueError("Die Anzahl der Merkmalsnamen passt nicht.")

# Die Aufteilung passiert vor jeder gelernten Transformation.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Diese Pipeline skaliert, wählt Merkmale und trainiert danach.
selection_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=5)),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

# Die Vergleichspipeline nutzt alle Merkmale ohne Auswahl.
full_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

# Beide Pipelines lernen ausschließlich aus den Trainingsdaten.
selection_pipeline.fit(X_train, y_train)
full_pipeline.fit(X_train, y_train)

# Die Testdaten bleiben bis zur Bewertung unangetastet.
selected_accuracy = accuracy_score(y_test, selection_pipeline.predict(X_test))
full_accuracy = accuracy_score(y_test, full_pipeline.predict(X_test))

# Wir lesen die in der Pipeline gewählten Merkmale aus.
selector = selection_pipeline.named_steps["selector"]
selected_names = feature_names[selector.get_support()]
selected_scores = selector.scores_[selector.get_support()]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Genauigkeit mit Auswahl: {selected_accuracy:.3f}")
print(f"Genauigkeit mit allen Merkmalen: {full_accuracy:.3f}")
print("Gewählte Merkmale: " + ", ".join(selected_names))

# Das Balkendiagramm zeigt die Auswahlwerte der behaltenen Merkmale.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(selected_names, selected_scores, color="steelblue")
ax.set_title("Auswahl in der Pipeline: behaltene Merkmale")
ax.set_xlabel("Merkmal")
ax.set_ylabel("ANOVA-Auswahlwert")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()



## **3. Modelle verständlich machen**

### **3.1. Koeffizienten verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_01.jpg?v=1787648053" width="250">



>* Koeffizienten zeigen Richtung und Stärke von Zusammenhängen
>* Sie beweisen keine Ursache-Wirkung-Beziehungen

>* Skalierung macht Koeffizienten erst vergleichbar
>* Kodierung bestimmt die Bedeutung kategorialer Koeffizienten

>* Korrelationen können Koeffizienten instabil machen
>* Mit Fachwissen und Prüfungen absichern



In [ ]:
#@title Python-Code - Koeffizienten verstehen

# Wir vergleichen Koeffizienten eines linearen Modells.
# Standardisierung macht Koeffizienten besser vergleichbar.
# Das Diagramm zeigt Richtung und Stärke.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Regressionsdatensatz aus scikit-learn.
diabetes = load_diabetes()
X = diabetes.data
y = diabetes.target
feature_names = diabetes.feature_names

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Wir trennen Trainingsdaten und Testdaten reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Die Pipeline standardisiert Merkmale und trainiert Ridge-Regression.
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)

# Nach Standardisierung sind Koeffizienten auf ähnlicher Skala.
ridge_model = model.named_steps["ridge"]
coefficients = ridge_model.coef_
score = model.score(X_test, y_test)

# Wir sortieren nach absoluter Koeffizientengröße.
order = np.argsort(np.abs(coefficients))
sorted_names = np.array(feature_names)[order]
sorted_coefficients = coefficients[order]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Test-R² des Modells: {score:.2f}")
print("Positive Werte erhöhen, negative senken die Vorhersage.")

# Ein Balkendiagramm zeigt Vorzeichen und Stärke gemeinsam.
colors = np.where(sorted_coefficients >= 0, "tab:blue", "tab:orange")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(sorted_names, sorted_coefficients, color=colors)

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Standardisierte Koeffizienten einer Ridge-Regression")
ax.set_xlabel("Koeffizient nach Standardisierung")
ax.set_ylabel("Merkmal")

plt.tight_layout()
plt.show()



### **3.2. Merkmalswichtigkeit verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_02.jpg?v=1787648052" width="250">



>* Wichtige Merkmale prägen Modellvorhersagen stark
>* Wichtigkeit bedeutet Signal, nicht Ursache

>* Wichtigkeitsmaße bedeuten je nach Verfahren Unterschiedliches
>* Korrelationen und Datenmerkmale können Werte verzerren

>* Wichtigkeit zeigt plausible oder problematische Modellsignale
>* Mit weiteren Erklärmethoden gemeinsam nutzen



In [ ]:
#@title Python-Code - Merkmalswichtigkeit verstehen

# Dieses Beispiel zeigt Merkmalswichtigkeit mit einem Baumensemble.
# Permutation misst den Leistungsverlust nach zufälligem Vertauschen.
# Der Balkenplot macht wichtige Merkmale sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

# Eine einfache Prüfung verhindert missverständliche Ergebnisse.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Split trennt Training und faire spätere Bewertung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Ein Random Forest liefert modellinterne Wichtigkeiten.
model = RandomForestClassifier(
    n_estimators=120, max_depth=5, random_state=42
)
model.fit(X_train, y_train)

# Die Testgenauigkeit zeigt die Ausgangsleistung des Modells.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit vor Permutation: {accuracy:.3f}")

# Permutation Importance misst den Leistungsverlust pro Merkmal.
result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

# Wir zeigen nur die fünf stärksten Merkmale.
top_indices = np.argsort(result.importances_mean)[-5:][::-1]
top_names = feature_names[top_indices]
top_scores = result.importances_mean[top_indices]
print(f"Wichtigstes Merkmal: {top_names[0]}")

# Der Plot vergleicht die wichtigsten Permutationswerte.
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_names[::-1], top_scores[::-1], color="steelblue")
ax.set_title("Top 5 Merkmale nach Permutation Importance")

# Die Achsenbeschriftung erklärt die Bedeutung der Werte.
ax.set_xlabel("Mittlerer Genauigkeitsverlust")
ax.set_ylabel("Merkmal")
plt.tight_layout()
plt.show()



### **3.3. Erklärungsprojekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_03.jpg?v=1787648055" width="250">



>* Erklärungskontext und Zielgruppe zuerst klären
>* Koeffizienten, Wichtigkeiten und Plots gemeinsam nutzen

>* Globale und lokale Erklärungen getrennt betrachten
>* Ranglisten durch Einzelfallanalysen ergänzen

>* Erklärungen kritisch und nicht kausal lesen
>* Mit Fachwissen, Datenprüfung und Validierung absichern



In [ ]:
#@title Python-Code - Erklärungsprojekt

# Dieses Projekt erklärt ein trainiertes Modell.
# Wir vergleichen Koeffizienten und Abhängigkeiten.
# Die Ausgabe zeigt wichtige Modellmuster.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.inspection import PartialDependenceDisplay
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Regressionsdatensatz aus scikit-learn.
diabetes = load_diabetes(as_frame=True)
features = diabetes.data
target = diabetes.target

# Diese Prüfung macht die Datenannahme sichtbar.
if features.shape[0] != target.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Wir teilen Daten, damit die Bewertung fairer bleibt.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Die Pipeline skaliert Merkmale und trainiert Ridge-Regression.
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)

# Koeffizienten zeigen Richtung und Stärke im linearen Modell.
ridge_model = model.named_steps["ridge"]
coefficients = pd.Series(ridge_model.coef_, index=features.columns)
top_coefficients = coefficients.abs().sort_values(ascending=False).head(3)

# Wir bewerten kurz, wie gut das Modell generalisiert.
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Test-R²: {r2:.2f}")
print("Drei stärkste Koeffizienten nach Betrag:")
for feature_name in top_coefficients.index:
    value = coefficients.loc[feature_name]
    print(f"{feature_name}: {value:.2f}")

# Der Plot zeigt die modellierte Abhängigkeit eines Merkmals.
important_feature = top_coefficients.index[0]
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(
    model, X_test, [important_feature], ax=ax
)

ax.set_title(f"Abhängigkeitsplot für {important_feature}")
ax.set_xlabel(f"Wert von {important_feature}")
ax.set_ylabel("Vorhergesagter Krankheitswert")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Merkmale erklären**</font>


In this lecture, you learned to:
- Konstruieren und wählen Merkmale innerhalb von scikit-learn-Pipelines aus. 
- Vergleichen Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen. 
- Interpretieren Modelle mit Koeffizienten, Importances und Abhängigkeitsplots. 

In the next Module (Module 12), we will go over 'Cluster und Text'